In [7]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

# SỬA ĐƯỜNG DẪN NÀY theo vị trí file trên máy em.
INPUT_PATH = Path(r"D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\01_data_cleaned.csv")
OUTPUT_DIR = Path("outputs/02_time_split_70_15_15")

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
VAL_SELECT_FRACTION = 0.60  # 60% số NGÀY Validation chọn cấu hình

COLUMNS = ["ticker", "date", "open", "high", "low", "close", "volume"]
NUMERIC = ["open", "high", "low", "close", "volume"]

if not np.isclose(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 1.0):
    raise ValueError("Tổng Train + Validation + Test phải bằng 1.0")
if not 0 < VAL_SELECT_FRACTION < 1:
    raise ValueError("VAL_SELECT_FRACTION phải nằm giữa 0 và 1")
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy file: {INPUT_PATH.resolve()}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [8]:
# Bản được gửi dùng ';', ngày DD/MM/YYYY. Notebook Bước 01 cũ
# lại xuất ',' và ngày YYYY-MM-DD: hỗ trợ cả HAI dạng, không đoán mơ hồ.
separator = None
for candidate in (";", ","):
    header = pd.read_csv(
        INPUT_PATH, sep=candidate, nrows=0, encoding="utf-8-sig"
    ).columns.str.strip().tolist()
    if len(header) == len(set(header)) and set(COLUMNS).issubset(header):
        separator = candidate
        break
if separator is None:
    raise ValueError(
        "Không tìm thấy 7 cột OHLCV. Kiểm tra đúng file và dấu phân cách."
    )

df = pd.read_csv(INPUT_PATH, sep=separator, encoding="utf-8-sig")
df.columns = df.columns.str.strip()
df = df[COLUMNS].copy()  # Bỏ cột ref_price hoặc metadata, KHÔNG dùng làm đầu vào

df["ticker"] = df["ticker"].astype("string").str.strip()

# Chỉ chấp nhận rõ hai định dạng ngày đã gặp trong dự án.
date_format = None
for candidate in ("%d/%m/%Y", "%Y-%m-%d"):
    try:
        parsed_dates = pd.to_datetime(df["date"], format=candidate, errors="raise")
        date_format = candidate
        break
    except (ValueError, TypeError):
        pass
if date_format is None:
    raise ValueError("Định dạng date không phải DD/MM/YYYY hoặc YYYY-MM-DD")
df["date"] = parsed_dates

for column in NUMERIC:
    df[column] = pd.to_numeric(df[column], errors="raise")

if df.empty:
    raise ValueError("File sạch không có bản ghi")
if df[COLUMNS].isna().any().any() or df["ticker"].eq("").any():
    raise ValueError("File sạch còn thiếu mã, ngày hoặc giá trị OHLCV")
if not np.isfinite(df[NUMERIC].to_numpy(dtype=float)).all():
    raise ValueError("Có số không hữu hạn (NaN/inf) trong OHLCV")
if df.duplicated(["ticker", "date"]).any():
    raise ValueError("File sạch còn trùng khóa ticker-date: quay lại Bước 01")
if not (df[["open", "high", "low", "close"]] > 0).all().all():
    raise ValueError("Giá OHLC không dương: quay lại Bước 01")
if not df["volume"].ge(0).all():
    raise ValueError("Volume âm: quay lại Bước 01")
if not df["high"].ge(df[["open", "low", "close"]].max(axis=1)).all():
    raise ValueError("Có giá High không hợp lệ: quay lại Bước 01")
if not df["low"].le(df[["open", "high", "close"]].min(axis=1)).all():
    raise ValueError("Có giá Low không hợp lệ: quay lại Bước 01")

df = df.sort_values(["date", "ticker"]).reset_index(drop=True)
print("Đọc thành công:", INPUT_PATH.name)
print("Dấu phân cách:", repr(separator), "| Ngày:", date_format)
print("Bản ghi:", len(df), "| Mã:", df["ticker"].nunique())


Đọc thành công: 01_data_cleaned.csv
Dấu phân cách: ';' | Ngày: %d/%m/%Y
Bản ghi: 15977 | Mã: 13


In [9]:
# Lấy tất cả ngày có ít nhất một mã giao dịch; không chia theo số dòng.
days = pd.DatetimeIndex(df["date"].drop_duplicates().sort_values())
n_days = len(days)
n_train = int(n_days * TRAIN_RATIO)
n_val = int(n_days * VAL_RATIO)
n_test = n_days - n_train - n_val
if min(n_train, n_val, n_test) < 2:
    raise ValueError("Không đủ ngày để chia thành các tập theo tỷ lệ")

train_end = days[n_train - 1]
val_end = days[n_train + n_val - 1]

df["split"] = np.select(
    [df["date"].le(train_end), df["date"].le(val_end)],
    ["train", "val"],
    default="test",
)

# Validation vẫn là MỘT tập; hai vai trò để tránh dùng lại tập
# chọn cấu hình để đồng thời điều chỉnh ngưỡng cảnh báo.
n_select = int(n_val * VAL_SELECT_FRACTION)
if not 0 < n_select < n_val:
    raise ValueError("Validation không đủ ngày để tách hai vai trò")
val_select_end = days[n_train + n_select - 1]
df["val_role"] = "not_applicable"
df.loc[(df["split"] == "val") & df["date"].le(val_select_end), "val_role"] = "select"
df.loc[(df["split"] == "val") & df["date"].gt(val_select_end), "val_role"] = "calibrate"


In [10]:
assert len(df) == len(df.drop_duplicates(["ticker", "date"]))
assert set(df["split"]) == {"train", "val", "test"}
assert df.groupby("date")["split"].nunique().eq(1).all()
assert df.groupby("date")["val_role"].nunique().eq(1).all()
assert df.loc[df["split"] != "val", "val_role"].eq("not_applicable").all()
assert df.loc[df["split"] == "val", "val_role"].isin(["select", "calibrate"]).all()
assert df.groupby("ticker")["split"].nunique().eq(3).all()

summary = (
    df.groupby("split")
    .agg(rows=("ticker", "size"), days=("date", "nunique"),
         first_date=("date", "min"), last_date=("date", "max"),
         tickers=("ticker", "nunique"))
    .reindex(["train", "val", "test"])
    .reset_index()
)
summary["row_ratio_pct"] = (100 * summary["rows"] / len(df)).round(2)

inventory = (
    df.groupby(["ticker", "split"]).size().unstack(fill_value=0)
    .reindex(columns=["train", "val", "test"], fill_value=0)
    .reset_index()
)

validation_roles = (
    df.loc[df["split"] == "val"]
    .groupby("val_role")
    .agg(rows=("date", "size"), days=("date", "nunique"),
         first_date=("date", "min"), last_date=("date", "max"))
    .reindex(["select", "calibrate"])
    .reset_index()
)

# Bảng lịch sử liên tục, đã gắn nhãn. Bước 03 phải tính rolling theo mã
# trên chuỗi liên tục; KHÔNG groupby(split) để rolling hoặc shift.
labeled = df[COLUMNS + ["split", "val_role"]].copy()
labeled["date"] = labeled["date"].dt.strftime("%Y-%m-%d")
manifest = labeled[["ticker", "date", "split", "val_role"]].copy()

labeled.to_csv(OUTPUT_DIR / "02_ohlcv_with_split.csv", index=False, encoding="utf-8-sig")
manifest.to_csv(OUTPUT_DIR / "02_split_manifest.csv", index=False, encoding="utf-8-sig")
summary.to_csv(OUTPUT_DIR / "02_split_summary.csv", index=False, encoding="utf-8-sig")
inventory.to_csv(OUTPUT_DIR / "02_split_inventory.csv", index=False, encoding="utf-8-sig")
validation_roles.to_csv(OUTPUT_DIR / "02_validation_roles.csv", index=False, encoding="utf-8-sig")

config = {
    "input_file": str(INPUT_PATH.resolve()),
    "input_sha256": hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest(),
    "input_separator": separator,
    "input_date_format": date_format,
    "source_columns_used": COLUMNS,
    "split_basis": "distinct observed market dates; same boundaries for all 13 tickers",
    "target_ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
    "days": {"total": n_days, "train": n_train, "val": n_val, "test": n_test},
    "boundaries": {
        "train_end": train_end.strftime("%Y-%m-%d"),
        "val_end": val_end.strftime("%Y-%m-%d"),
        "val_select_end": val_select_end.strftime("%Y-%m-%d"),
    },
    "val_select_fraction_by_days": VAL_SELECT_FRACTION,
    "feature_engineering_rule": "causal per-ticker rolling on continuous history, never restart at split boundaries",
}
(OUTPUT_DIR / "02_split_config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("\n=== TRAIN / VALIDATION / TEST ===")
print(summary.to_string(index=False))
print("\n=== HAI VAI TRÒ TRONG VALIDATION ===")
print(validation_roles.to_string(index=False))
print("\nĐã lưu kết quả vào:", OUTPUT_DIR.resolve())
print("Bước 02 hoàn tất; chưa tạo đặc trưng hoặc chạy mô hình.")



=== TRAIN / VALIDATION / TEST ===
split  rows  days first_date  last_date  tickers  row_ratio_pct
train 11180   861 2021-10-19 2025-04-01       13          69.98
  val  2392   184 2025-04-02 2025-12-23       13          14.97
 test  2405   185 2025-12-24 2026-09-25       13          15.05

=== HAI VAI TRÒ TRONG VALIDATION ===
 val_role  rows  days first_date  last_date
   select  1430   110 2025-04-02 2025-09-10
calibrate   962    74 2025-09-11 2025-12-23

Đã lưu kết quả vào: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\02_time_split_70_15_15
Bước 02 hoàn tất; chưa tạo đặc trưng hoặc chạy mô hình.
